In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score

URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(URL)

print(df.shape)
df.head()

(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
# Dataset
# In this homework, we will use the 2026 lead-scoring dataset. Download it from here.

# Or you can do it with wget:

# wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
# In this dataset our desired target for classification task will be converted variable - has the client signed up to the platform or not.
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')
# Data preparation
# Check if the missing values are presented in the features.
# If there are missing values:
# For categorical features, replace them with 'NA'
# For numerical features, replace with with 0.0
# annual_income para 0.0
# location to NA
df.isnull().sum()
target = "converted"

# O tipo é detectado pelo conteúdo da coluna (funciona em qualquer versão do pandas)
numerical = [c for c in df.columns
             if c != target and pd.api.types.is_numeric_dtype(df[c])]
categorical = [c for c in df.columns
               if c != target and c not in numerical]

print("categóricas:", categorical)
print("numéricas  :", numerical)

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print("valores ausentes restantes:", int(df.isnull().sum().sum()))  # tem que ser 0
# Question 1
# What is the most frequent observation (mode) for the column industry?

# NA
# technology - tech
# healthcare
# retail
print(df["industry"].value_counts())

ans1 = df["industry"].mode()[0]
print("\nResposta Q1:", ans1)

categóricas: ['lead_source', 'industry', 'employment_status', 'location']
numéricas  : ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
valores ausentes restantes: 0
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Resposta Q1: technology


In [4]:
# Question 2
# Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.
corr = df[numerical].corr()
corr.round(3)
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

ranking = pd.DataFrame([(a, b, corr.loc[a, b]) for a, b in pairs],
                       columns=["feature_1", "feature_2", "r"])
ranking["|r|"] = ranking["r"].abs()
ranking = ranking.sort_values("|r|", ascending=False).reset_index(drop=True)
print(ranking.round(4).to_string(index=False))

res2 = (ranking.loc[0, "feature_1"], ranking.loc[0, "feature_2"])
print(res2)
# What are the two features that have the biggest correlation?

# interaction_count and lead_score
# number_of_courses_viewed and lead_score
# number_of_courses_viewed and interaction_count
# annual_income and interaction_count
# Only consider the pairs above when answering this question.


               feature_1         feature_2      r    |r|
       interaction_count        lead_score 0.9157 0.9157
number_of_courses_viewed        lead_score 0.7572 0.7572
number_of_courses_viewed interaction_count 0.7216 0.7216
           annual_income interaction_count 0.1228 0.1228
('interaction_count', 'lead_score')


In [10]:
# Split the data
# Split the data with these exact calls:
# df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
# df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
# Make sure that the target value converted is not in your dataframe.

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

df_train = df_train.drop(columns=[target])
df_val = df_val.drop(columns=[target])
df_test = df_test.drop(columns=[target])

print('treino:', df_train.shape, "| validação:", df_val.shape, "| teste:", df_test.shape)
print('converted' ' ainda está nas colunas?', target in df_train.columns)  # False

treino: (3000, 8) | validação: (1000, 8) | teste: (1000, 8)
converted ainda está nas colunas? False


In [6]:
mi_exact = {col: mutual_info_score(df_train[col], y_train) for col in categorical}
mi_round = {col: round(score, 2) for col, score in mi_exact.items()}

mi_table = (pd.DataFrame({'mi_arredondada': mi_round, 'mi_exata': mi_exact})
              .sort_values('mi_exata', ascending=False))
print(mi_table.round(4))

# maior valor arredondado; se empatar, a versão exata desempata
res3 = max(mi_round, key=lambda c: (mi_round[c], mi_exact[c]))
print(res3)

if list(mi_round.values()).count(mi_round[res3]) > 1:
    print('empate depois de arredondar')
# Question 3
# Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
# Round the scores to 2 decimals using round(score, 2).
# Which of these variables has the biggest mutual information score?

# industry
# location
# lead_source
# mployment_status



                   mi_arredondada  mi_exata
lead_source                  0.03    0.0343
employment_status            0.02    0.0190
industry                     0.00    0.0020
location                     0.00    0.0010
lead_source


In [7]:
features = categorical + numerical  # todas as variáveis explicativas
def train_and_eval(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[feats].to_dict(orient='records'))
    X_val = dv.transform(df_val[feats].to_dict(orient='records'))  # transform, NÃO fit_transform

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    return float((y_pred == y_val).mean())


acc_base = train_and_eval(features)
res4 = round(acc_base, 2)

print(acc_base)
print(res4)
# Question 4
# Now let's train a logistic regression.
# Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
# Fit the model on the training dataset.
# To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
# model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
# What accuracy did you get?

# 0.55
# 0.65
# 0.75
# 0.85

0.645
0.65


In [8]:
acc_without = {}
diffs = {}

for f in features:
    remaining = [x for x in features if x != f]
    acc_without[f] = train_and_eval(remaining)
    diffs[f] = acc_base - acc_without[f]

elim = (pd.DataFrame({'acc_sem_a_variavel': acc_without, 'diferenca': diffs})
          .sort_values('diferenca'))
print(f'acc original: {acc_base:.6f}\n')
print(elim.round(6))

options_q5 = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
res5 = min(options_q5, key=lambda f: diffs[f])
print(res5)
# Question 5
# Let's find the least useful feature using the feature elimination technique.
# Train a model using the same features and parameters as in Q4 (without rounding).
# Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
# For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
# Which of following feature has the smallest difference?

# 'lead_source'
# 'number_of_courses_viewed'
# 'interaction_count'
# Note: The difference doesn't have to be positive.

acc original: 0.645000

                          acc_sem_a_variavel  diferenca
annual_income                          0.724     -0.079
industry                               0.645      0.000
location                               0.645      0.000
employment_status                      0.645      0.000
lead_score                             0.644      0.001
number_of_courses_viewed               0.643      0.002
lead_source                            0.642      0.003
interaction_count                      0.601      0.044
number_of_courses_viewed


In [9]:
Cs = [0.000001, 0.00001, 0.0001, 0.001]

acc_by_C = {C: round(train_and_eval(features, C=C), 3) for C in Cs}
for C, acc in acc_by_C.items():
    print(f'C = {C:<9g} acc = {acc:.3f}')

best_acc = max(acc_by_C.values())
res6 = min(C for C, acc in acc_by_C.items() if acc == best_acc)  # empate -> menor C

baseline = max(y_val.mean(), 1 - y_val.mean())
print({baseline:.3})
print(res6)
# Question 6
# Now let's train a regularized logistic regression.
#Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
#T rain models using all the features as in Q4.
# C alculate the accuracy on the validation dataset and round it to 3 decimal digits.
# Which of these C leads to the best accuracy on the validation set?

# 0.000001
# 0.00001
# 0.0001
# 0.001

C = 1e-06     acc = 0.598
C = 1e-05     acc = 0.598
C = 0.0001    acc = 0.613
C = 0.001     acc = 0.645
{np.float64(0.597): 0.3}
0.001
